In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans

def engineer_master_features(train_df, test_df):
    print("Combining for rolling features...")
    train_df['is_test'] = 0
    test_df['is_test'] = 1
    test_df['demand'] = np.nan

    combined = pd.concat([train_df, test_df], ignore_index=True)

    # Extract Hour for grouping
    combined['hour'] = combined['timestamp'].apply(lambda x: int(str(x).split(':')[0]) if pd.notnull(x) else np.nan)

    # ==========================================
    # UPGRADE A: Historical Spatio-Temporal Target Encoding
    # ==========================================
    # Calculate historical mean demand ONLY on the training data to prevent leakage
    train_only = combined[combined['is_test'] == 0]

    # 1. Overall average for the geohash
    geo_means = train_only.groupby('geohash')['demand'].mean().rename('hist_geo_mean')

    # 2. Average for the geohash at this specific hour
    geo_hour_means = train_only.groupby(['geohash', 'hour'])['demand'].mean().rename('hist_geo_hour_mean')

    # Map back to the combined dataset
    combined = combined.merge(geo_means, on='geohash', how='left')
    combined = combined.merge(geo_hour_means, on=['geohash', 'hour'], how='left')

    # Fill missing historicals with the global average demand
    global_mean = train_only['demand'].mean()
    combined['hist_geo_mean'] = combined['hist_geo_mean'].fillna(global_mean)
    combined['hist_geo_hour_mean'] = combined['hist_geo_hour_mean'].fillna(global_mean)

    # ==========================================
    # UPGRADE B: Rolling Window Statistics (Momentum)
    # ==========================================
    combined = combined.sort_values(['geohash', 'timestamp', 'day'])

    # Calculate the rolling mean and standard dev over the last 3 time steps
    # We must shift(1) first so we don't include the current row's demand!
    rolling_groups = combined.groupby('geohash')['demand'].shift(1)

    combined['rolling_mean_3'] = combined.groupby('geohash')['demand'].shift(1).rolling(window=3, min_periods=1).mean()
    combined['rolling_std_3']  = combined.groupby('geohash')['demand'].shift(1).rolling(window=3, min_periods=1).std()

    # Fill NaNs for the first few rows of a geohash
    combined['rolling_mean_3'] = combined['rolling_mean_3'].fillna(-1)
    combined['rolling_std_3'] = combined['rolling_std_3'].fillna(-1)

    combined = combined.sort_index() # Restore original order

    # ==========================================
    # UPGRADE C: K-Means Spatial Clustering
    # ==========================================
    print("Clustering Spatial Neighborhoods...")
    import pygeohash as pgh
    combined['lat'] = combined['geohash'].apply(lambda x: pgh.decode(x)[0] if pd.notnull(x) else np.nan)
    combined['lon'] = combined['geohash'].apply(lambda x: pgh.decode(x)[1] if pd.notnull(x) else np.nan)

    # We cluster based on lat/lon coordinates
    coords = combined[['lat', 'lon']].dropna()

    # Create 30 distinct spatial zones
    kmeans = KMeans(n_clusters=30, random_state=42, n_init=10)
    kmeans.fit(coords)

    # Map the clusters back to the dataframe
    combined.loc[coords.index, 'spatial_cluster'] = kmeans.labels_
    combined['spatial_cluster'] = combined['spatial_cluster'].fillna(-1).astype(int)

    # Split back to train and test
    train_final = combined[combined['is_test'] == 0].drop(columns=['is_test'])
    test_final = combined[combined['is_test'] == 1].drop(columns=['is_test', 'demand'])

    return train_final, test_final